In [1]:
import pandas as pd

df = pd.read_csv("datatelco_customer_churn.csv")

pd.to_numeric(df['TotalCharges'], errors='coerce')
df.head()
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

# El problema con las columnas que no son reconocidas directamente como numéricas

Luego de investigar un poco, descubrimos que Pandas infiere el dtype al leer el CSV, columna por columna, y aplica una regla simple: si todos los valores de la columna se pueden convertir a número, la hace numérica pero si aparece aunque sea uno solo que no puede, toda la columna cae a object.Object significa que la columna no guarda los valores en sí, sino punteros a objetos de Python. En la práctica, casi siempre son strings.

El caso que encontramos fue con TotalCharges. Hay 7043 filas, de las cuales 7032 son montos perfectamente numéricos. Pero 11 tienen un espacio en blanco. Ese espacio no se puede convertir a número, así que pandas se rinde y guarda las 7043 como strings. Un "29.85" con comillas, no un 29.85.

Los espacios en blanco aparecen cuando son clientes con tenure = 0, o sea que recién se dieron de alta y todavía no facturaron nada. Esa columna sí es numérica conceptualmente, pero pandas la lee como object porque hay unas 11 filas con un string vacío (" ") en lugar de un número.

El impacto de no detectarlo es que la columna queda como texto y el modelo o la ignora, o si alguien la codifica como categórica, termina tratando cada monto como una categoría distinta.

Se arregla con pd.to_numeric(df['TotalCharges'], errors='coerce') y después tendríamos que decidir qué hacer con esos NaN (imputar con 0 tiene sentido acá, justificando que no facturaron todavía).

In [2]:
# 1. Conversión: los " " no parseables pasan a NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Verificación del diagnóstico (para dejar registro en el notebook)
faltantes = df['TotalCharges'].isna()
print(f"Filas con TotalCharges nulo: {faltantes.sum()}")
print(f"Valores de tenure en esas filas: {df.loc[faltantes, 'tenure'].unique()}")

# 3. Chequeo final
print(f"\nDtype: {df['TotalCharges'].dtype}")
print(f"Nulos restantes: {df['TotalCharges'].isna().sum()}")

df.dtypes

Filas con TotalCharges nulo: 11
Valores de tenure en esas filas: [0]

Dtype: float64
Nulos restantes: 11


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

## 2. Preprocesamiento y prevención de Data Leakage

### **Valores faltantes**

Compruebo que el único problema de datos faltantes en todo el dataset fue el de TotalCharges, lo cual se deja constancia a continuación.

In [3]:
# Nulos "reales" en todas las columnas
print(df.isnull().sum()[df.isnull().sum() > 0])

# Nulos compuestos por "texto raro" en columnas categóricas
for col in df.select_dtypes(include='object').columns:
    valores_raros = df[col].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
    if valores_raros > 0:
        print(f"{col}: {valores_raros} valores vacíos/espacios")

TotalCharges    11
dtype: int64


Para las filas con `TotalCharges` nulo y `tenure=0`, imputo con 0, ya que son clientes que recién se dieron de alta y todavía no generaron facturación acumulada.

Si quedaran filas con `TotalCharges` nulo y `tenure>0` (verificado arriba), esos casos se imputan luego (para evitar data leakage)

In [4]:
# Imputación condicional (justificada por lógica de negocio)
condicion_tenure_cero = (df['TotalCharges'].isna()) & (df['tenure'] == 0)
condicion_otros_nulos = (df['TotalCharges'].isna()) & (df['tenure'] != 0)

print(f"Nulos con tenure=0: {condicion_tenure_cero.sum()}")
print(f"Nulos con tenure>0: {condicion_otros_nulos.sum()}")

df.loc[condicion_tenure_cero, 'TotalCharges'] = 0

print(f"Nulos restantes en TotalCharges: {df['TotalCharges'].isna().sum()}")

Nulos con tenure=0: 11
Nulos con tenure>0: 0
Nulos restantes en TotalCharges: 0


In [5]:
# Imputación para el caso tenure=0: TotalCharges = 0 (lógica de negocio, el cliente todavía no facturó nada)
df.loc[condicion_tenure_cero, 'TotalCharges'] = 0

# Chequeo: cuántos nulos quedan sin resolver (los de tenure>0, si los hubiera)
print(f"Nulos restantes en TotalCharges: {df['TotalCharges'].isna().sum()}")

Nulos restantes en TotalCharges: 0


### **Variables categóricas**

Al revisar las categorías de las variables de texto, encontramos que varias columnas no son binarias en apariencia, pero sí lo son en esencia:

- `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`,`StreamingMovies` tienen 3 valores: `Yes`, `No`, `No internet service`.
- `MultipleLines` tiene 3 valores: `Yes`, `No`, `No phone service`.

El problema es que la tercera categoría no aporta información nueva: un cliente tiene `"No internet service"` en esas 6 columnas si y solo si `InternetService == "No"`, y tiene `"No phone service"` en `MultipleLines` si y solo si `PhoneService == "No"`. Es decir, esa categoría es perfectamente redundante con una variable que ya está en el dataset.

Por eso, decidimos colapsar `"No internet service"` y `"No phone service"` en `"No"`. Esto convierte a las 7 columnas mencionadas en verdaderamente binarias, permitiendo aplicar Label Encoding de forma consistente con el resto de las variables Yes/No, y sin perder información (porque esa información ya vive en `InternetService` y `PhoneService`).

In [6]:
# Colapsamos "No internet service" / "No phone service" en "No"
cols_no_internet = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_no_internet:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Verificación
for col in cols_no_internet + ['MultipleLines']:
    print(f"{col}: {df[col].unique()}")

OnlineSecurity: ['No' 'Yes']
OnlineBackup: ['Yes' 'No']
DeviceProtection: ['No' 'Yes']
TechSupport: ['No' 'Yes']
StreamingTV: ['No' 'Yes']
StreamingMovies: ['No' 'Yes']
MultipleLines: ['No' 'Yes']


In [7]:
#Ahora aplico One Hot Encoding y Label Encoding a las columnas categóricas
from sklearn.preprocessing import LabelEncoder

# customerID es un identificador único por cliente, no una variable predictiva. Si no hacemos este paso, 
# el One-Hot Encoding lo trataría como una columna categórica
df = df.drop(columns=['customerID'])

# 1. Separamos las columnas categóricas en binarias (2 categorías) y multiclase (3+),
# dejando Churn afuera porque es el target y lo tratamos aparte
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Churn')

binarias = [col for col in cat_cols if df[col].nunique() == 2]
multiclase = [col for col in cat_cols if df[col].nunique() > 2]

#Verificación
print("Binarias (Label Encoding):", binarias)
print("Multiclase (One-Hot Encoding):", multiclase)

Binarias (Label Encoding): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']
Multiclase (One-Hot Encoding): ['InternetService', 'Contract', 'PaymentMethod']


In [8]:
# 2. Label Encoding para las binarias
le = LabelEncoder()
mapeos = {}  # guardamos el mapeo de cada columna para dejar constancia

for col in binarias:
    df[col] = le.fit_transform(df[col])
    mapeos[col] = dict(zip(le.classes_, le.transform(le.classes_)))

for col, mapeo in mapeos.items():
    print(f"{col}: {mapeo}")

# 3. One-Hot Encoding para las que tienen 3 o más categorías 
df = pd.get_dummies(df, columns=multiclase, drop_first=True)

# 4. Encodeamos el target por separado (No -> 0, Yes -> 1)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# 5. Chequeo final
print(df.dtypes.value_counts())
print(df.select_dtypes(include='object').columns)

# Convertimos columnas booleanas a enteros para mejor compatibilidad con modelos de ML
dummy_cols = df.select_dtypes(include='bool').columns
df[dummy_cols] = df[dummy_cols].astype(int)

gender: {'Female': np.int64(0), 'Male': np.int64(1)}
Partner: {'No': np.int64(0), 'Yes': np.int64(1)}
Dependents: {'No': np.int64(0), 'Yes': np.int64(1)}
PhoneService: {'No': np.int64(0), 'Yes': np.int64(1)}
MultipleLines: {'No': np.int64(0), 'Yes': np.int64(1)}
OnlineSecurity: {'No': np.int64(0), 'Yes': np.int64(1)}
OnlineBackup: {'No': np.int64(0), 'Yes': np.int64(1)}
DeviceProtection: {'No': np.int64(0), 'Yes': np.int64(1)}
TechSupport: {'No': np.int64(0), 'Yes': np.int64(1)}
StreamingTV: {'No': np.int64(0), 'Yes': np.int64(1)}
StreamingMovies: {'No': np.int64(0), 'Yes': np.int64(1)}
PaperlessBilling: {'No': np.int64(0), 'Yes': np.int64(1)}
int64      15
bool        7
float64     2
Name: count, dtype: int64
Index([], dtype='object')


### Feature scaling
Utilizamos StandardScaler (completar por qué)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separamos features y target
X = df.drop(columns=['Churn'])
y = df['Churn']

# 2. Train/Test split ANTES de escalar, para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Columnas numéricas continuas a escalar (las binarias/dummies ya son 0/1, no hace falta)
cols_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']

# 4. Fiteamos el scaler SOLO con X_train
scaler = StandardScaler()
X_train[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])
X_test[cols_numericas] = scaler.transform(X_test[cols_numericas])  # solo transform, sin fit

X_train[cols_numericas].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00
